<a href="https://colab.research.google.com/github/Comfy05/Football_analytics_portfolio/blob/main/05_model_xG/xG_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install mplsoccer
!pip install statsbombpy

from pandas.core.arrays import period
import pandas as pd
from statsbombpy import sb
from mplsoccer import Pitch, VerticalPitch
from matplotlib.pyplot import scatter
import matplotlib.pyplot as plt
import scipy as sc
from scipy.ndimage import gaussian_filter
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import class_likelihood_ratios

In [9]:
competitions = sb.competitions()

competiotions_WC2018 = competitions['competition_name'] == 'FIFA World Cup'
competiotions_WC2018 = competitions[competiotions_WC2018]

matches_WC2018 = sb.matches(competition_id=43, season_id=3)

In [10]:
xG_list = []

for index, row in matches_WC2018.iterrows():
  df_xG = sb.events(match_id = row['match_id'])
  xG_bool = df_xG['type'] == 'Shot'
  df_xG = df_xG[xG_bool][['player', 'team', 'match_id', 'location', 'shot_outcome', 'shot_body_part', 'shot_statsbomb_xg']]
  xG_list.append(df_xG)

xG_final = pd.concat(xG_list, axis=0, ignore_index=True)
xG_final = xG_final.join(pd.DataFrame(xG_final.pop('location').tolist(), index = xG_final.index, columns=["x", 'y']))
#Pitagoras equation
xG_final['distance'] = np.sqrt((120-xG_final['x'])**2 +(40-xG_final['y'])**2)
#cosinus equation
xG_final['angle'] = np.arccos((((120-xG_final['x'])**2 +(36-xG_final['y'])**2) + ((120-xG_final['x'])**2 +(44-xG_final['y'])**2) - (8**2)) / (2 * np.sqrt((120-xG_final['x'])**2 +(36-xG_final['y'])**2) * np.sqrt((120-xG_final['x'])**2 +(44-xG_final['y'])**2))) * (180/np.pi)

xG_final['is_goal'] = xG_final['shot_outcome'] == 'Goal'
xG_final['is_goal'] = xG_final['is_goal'].astype(int)

body_part_encoded = pd.get_dummies(xG_final['shot_body_part'])
xG_final = xG_final.join(body_part_encoded)

In [11]:
X = xG_final[['distance', 'angle', 'Head', 'Left Foot','Other', 'Right Foot']]
y = xG_final['is_goal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train, X_test, y_train, y_test)

       distance      angle   Head  Left Foot  Other  Right Foot
590   24.515301  18.170101  False       True  False       False
1549  12.649111  33.690068   True      False  False       False
420    9.219544  39.062584   True      False  False       False
425   27.459060  13.999703  False      False  False        True
857   16.155494  11.084891  False       True  False       False
...         ...        ...    ...        ...    ...         ...
1130  11.313708  29.744881   True      False  False       False
1294  35.468296  12.000210  False      False  False        True
860    7.810250  41.633539  False       True  False       False
1459  34.132096  13.318318  False      False  False        True
1126  12.041595  26.387115   True      False  False       False

[1364 rows x 6 columns]        distance      angle   Head  Left Foot  Other  Right Foot
567   21.400935  17.102729  False      False  False        True
1324  26.248809  13.373313  False       True  False       False
1631  11.045361

In [12]:
model_xG = LogisticRegression().fit(X_train, y_train)
#uczenie maszynowe prawdopodobieństwa
y_proba = model_xG.predict_proba(X_test)

my_xG = y_proba[:,1]
my_xG

array([0.07597693, 0.0308895 , 0.28896199, 0.0286729 , 0.05225759,
       0.06316876, 0.09792582, 0.05559379, 0.28896199, 0.04504603,
       0.08731016, 0.18969849, 0.20355778, 0.04065323, 0.24988408,
       0.02249934, 0.21886067, 0.08224299, 0.01734259, 0.0238606 ,
       0.40697346, 0.10624042, 0.12484989, 0.22789344, 0.01780001,
       0.02197896, 0.11971822, 0.06281197, 0.31653745, 0.10220448,
       0.15572091, 0.02768921, 0.04113177, 0.07439996, 0.0712225 ,
       0.04132832, 0.06306212, 0.06054024, 0.01715835, 0.01942417,
       0.05078486, 0.07220144, 0.1109865 , 0.17135033, 0.03881287,
       0.12058762, 0.0525947 , 0.05225759, 0.08830592, 0.02779475,
       0.29873217, 0.28896199, 0.03325137, 0.10388996, 0.02704382,
       0.10465726, 0.03358541, 0.12655661, 0.0495796 , 0.11128819,
       0.0525947 , 0.03757976, 0.10939604, 0.09183378, 0.13322456,
       0.0539952 , 0.03250632, 0.04881422, 0.26901829, 0.35339035,
       0.06994454, 0.04437708, 0.10374272, 0.0238606 , 0.30686

In [13]:
y_proba = model_xG.predict_proba(X)

my_xG = y_proba[:,1]
xG_final['my_xG'] = my_xG
xG_final[[ 'player', 'team', 'match_id', 'shot_outcome', 'shot_body_part', 'shot_statsbomb_xg', 'my_xG']]

,player,team,match_id,shot_outcome,shot_body_part,shot_statsbomb_xg,my_xG
0,Kevin De Bruyne,Belgium,8650,Off T,Right Foot,0.020461,0.027904
1,Thiago Emiliano da Silva,Brazil,8650,Post,Right Foot,0.274021,0.642430
2,Eden Hazard,Belgium,8650,Blocked,Right Foot,0.065900,0.133747
3,Nacer Chadli,Belgium,8650,Off T,Right Foot,0.022908,0.030841
4,José Paulo Bezzera Maciel Júnior,Brazil,8650,Blocked,Right Foot,0.174216,0.224577
...,...,...,...,...,...,...,...
1701,Amr Medhat Warda,Egypt,7559,Blocked,Right Foot,0.028970,0.046882
1702,Mohammed Al Burayk,Saudi Arabia,7559,Blocked,Right Foot,0.065623,0.043550
1703,Salem Mohammed Al Dawsari,Saudi Arabia,7559,Saved,Right Foot,0.088908,0.196412
1704,Abdullah Ibrahim Otayf,Saudi Arabia,7559,Blocked,Right Foot,0.020325,0.043716
